# DentaScribe — Data Engineering
## Role 3 — Varsha | Individual Role Notebook

| | |
|---|---|
| **Author** | Varsha (Role 3 — Data Engineering) |
| **Course** | Deep Learning Group Project |
| **Scope** | Audio dataset acquisition, preprocessing, and delivery for Role 1 (ASR) |
| **Team notebook** | See `dentascribe_final_v3.ipynb` for all roles combined |

---

### What this notebook does
1. **Section 1** — Dataset overview and download instructions
2. **Section 2** — Audio loading and format validation
3. **Section 3** — Preprocessing pipeline (stereo→mono, resample 16 kHz, normalise)
4. **Section 4** — Train / validation / test split (80 / 10 / 10)
5. **Section 5** — Exploratory data analysis (duration, sample rate, label distribution)
6. **Section 6** — Export clean dataset for ASR training

> **Output:** Cleaned, 16 kHz mono WAV files + `train.csv` / `val.csv` / `test.csv` ready for Whisper fine-tuning


In [ ]:
!pip install librosa soundfile pandas matplotlib seaborn scikit-learn -q

## Section 1 — Dataset Overview

Two Kaggle datasets are used for fine-tuning the Whisper ASR model:

| Dataset | Size | Content |
|---|---|---|
| Medical Speech, Transcription & Intent | ~8.5 hrs | Medical symptom audio + transcripts |
| Multimodal Speech Recognition & Acoustic Feature | Variable | General speech + acoustic features |

**Download instructions:**
1. Install the Kaggle CLI: `pip install kaggle`
2. Place your `kaggle.json` API key in `~/.kaggle/`
3. Run the cells below to download both datasets


In [ ]:
import os, json
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

MEDICAL_SPEECH_DIR = 'datasets/medical_speech'
MULTIMODAL_DIR     = 'datasets/multimodal_asr'
OUTPUT_DIR         = 'datasets/processed'
TARGET_SR          = 16000

for d in [MEDICAL_SPEECH_DIR, MULTIMODAL_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')


In [ ]:
# Download Medical Speech dataset
# !kaggle datasets download -d paultimothymooney/medical-speech-transcription-and-intent
# !unzip medical-speech-transcription-and-intent.zip -d datasets/medical_speech

# Download Multimodal ASR dataset
# !kaggle datasets download -d imsparsh/multimodal-speech-recognition-acoustic-feature
# !unzip multimodal-speech-recognition-acoustic-feature.zip -d datasets/multimodal_asr

print('Uncomment above lines to download datasets.')


## Section 2 — Audio Loading and Format Validation

In [ ]:
def load_and_validate_audio(filepath):
    """Load an audio file and return its properties.

    Args:
        filepath (str): Path to the audio file.

    Returns:
        dict: Keys are path, duration_s, sample_rate, channels, valid.
    """
    try:
        info = sf.info(filepath)
        return {'path': filepath, 'duration_s': round(info.duration, 2),
                'sample_rate': info.samplerate, 'channels': info.channels, 'valid': True}
    except Exception as e:
        return {'path': filepath, 'valid': False, 'error': str(e)}

def scan_dataset(directory, extensions=('.wav', '.mp3', '.flac')):
    """Recursively scan a directory and validate all audio files.

    Args:
        directory (str): Root directory to scan.
        extensions (tuple[str]): Audio file extensions to include.

    Returns:
        pd.DataFrame: One row per audio file with validation metadata.
    """
    records = [load_and_validate_audio(str(fp))
               for fp in Path(directory).rglob('*')
               if fp.suffix.lower() in extensions]
    df = pd.DataFrame(records)
    print(f'Found {len(df)} files | Valid: {df["valid"].sum()} | Invalid: {(~df["valid"]).sum()}')
    return df

# df_medical    = scan_dataset(MEDICAL_SPEECH_DIR)
# df_multimodal = scan_dataset(MULTIMODAL_DIR)
print('[Section 2] Validation functions defined.')


## Section 3 — Preprocessing Pipeline

Each audio file is:
1. Converted stereo → **mono** (channel average)
2. Resampled to **16,000 Hz** (Whisper requirement)
3. **Normalised** to peak amplitude of 1.0
4. Saved as **16-bit PCM WAV**


In [ ]:
def preprocess_audio(input_path, output_path, target_sr=16000):
    """Preprocess a single audio file for Whisper ASR training.

    Args:
        input_path (str): Path to the source audio file.
        output_path (str): Destination path for the processed WAV.
        target_sr (int): Target sample rate in Hz. Defaults to 16000.

    Returns:
        bool: True if preprocessing succeeded, False otherwise.
    """
    try:
        audio, _ = librosa.load(input_path, sr=target_sr, mono=True)
        audio    = audio / (np.max(np.abs(audio)) + 1e-9)
        sf.write(output_path, audio, target_sr, subtype='PCM_16')
        return True
    except Exception as e:
        print(f'  Error: {input_path}: {e}')
        return False

def preprocess_dataset(df, output_dir, target_sr=16000):
    """Batch preprocess all valid audio files.

    Args:
        df (pd.DataFrame): DataFrame with a 'path' column from scan_dataset.
        output_dir (str): Directory for processed files.
        target_sr (int): Target sample rate. Defaults to 16000.

    Returns:
        pd.DataFrame: Input DataFrame with added 'processed_path' column.
    """
    os.makedirs(output_dir, exist_ok=True)
    rows = df[df['valid'] == True].copy()
    paths = []
    for _, row in rows.iterrows():
        out = os.path.join(output_dir, Path(row['path']).stem + '_16k.wav')
        paths.append(out if preprocess_audio(row['path'], out, target_sr) else None)
    rows['processed_path'] = paths
    print(f'Preprocessed {sum(p is not None for p in paths)}/{len(rows)} files.')
    return rows

# df_processed = preprocess_dataset(pd.concat([df_medical, df_multimodal]), OUTPUT_DIR)
print('[Section 3] Preprocessing functions defined.')


## Section 4 — Train / Validation / Test Split

Split ratio: **80% train | 10% validation | 10% test**

In [ ]:
def split_and_save(df, output_dir, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Split dataset into train/val/test and save to CSV.

    Args:
        df (pd.DataFrame): Preprocessed dataset.
        output_dir (str): Directory to write CSV files.
        train_ratio (float): Training proportion. Defaults to 0.8.
        val_ratio (float): Validation proportion. Defaults to 0.1.
        seed (int): Random seed. Defaults to 42.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: train, val, test.
    """
    test_ratio = round(1.0 - train_ratio - val_ratio, 2)
    train, temp = train_test_split(df, test_size=(1 - train_ratio), random_state=seed)
    val, test   = train_test_split(temp, test_size=test_ratio/(val_ratio+test_ratio), random_state=seed)
    os.makedirs(output_dir, exist_ok=True)
    train.to_csv(f'{output_dir}/train.csv', index=False)
    val.to_csv(f'{output_dir}/val.csv',     index=False)
    test.to_csv(f'{output_dir}/test.csv',   index=False)
    print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')
    print(f'Saved to {output_dir}/')
    return train, val, test

# train, val, test = split_and_save(df_processed, OUTPUT_DIR)
print('[Section 4] Split function defined.')


## Section 5 — Exploratory Data Analysis

In [ ]:
def plot_eda(df, title='ASR Dataset EDA'):
    """Plot audio duration and sample rate distributions.

    Args:
        df (pd.DataFrame): DataFrame with duration_s and sample_rate columns.
        title (str): Plot title. Defaults to 'ASR Dataset EDA'.

    Returns:
        None
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    axes[0].hist(df['duration_s'].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[0].set(xlabel='Duration (s)', ylabel='Count', title='Clip Duration Distribution')
    sr_counts = df['sample_rate'].value_counts()
    axes[1].bar(sr_counts.index.astype(str), sr_counts.values, color='coral', edgecolor='white')
    axes[1].set(xlabel='Sample Rate (Hz)', ylabel='Count', title='Sample Rate Breakdown')
    plt.tight_layout()
    plt.show()
    total = df['duration_s'].sum() / 3600
    print(f'Total: {total:.2f} hrs | Mean clip: {df["duration_s"].mean():.1f}s | Median: {df["duration_s"].median():.1f}s')

# plot_eda(df_processed, title='DentaScribe ASR Dataset')
print('[Section 5] EDA function defined.')


## Section 6 — Export Clean Dataset for ASR Training

Runs the complete pipeline and writes:
- `datasets/processed/train.csv`
- `datasets/processed/val.csv`
- `datasets/processed/test.csv`

> These files are passed directly to **Role 1 (Andy)** for Whisper fine-tuning.


In [ ]:
# ── Run full pipeline ───────────────────────────────────────────
# Uncomment after downloading both Kaggle datasets

# df_medical    = scan_dataset(MEDICAL_SPEECH_DIR)
# df_multimodal = scan_dataset(MULTIMODAL_DIR)
# df_all        = pd.concat([df_medical, df_multimodal], ignore_index=True)
# df_processed  = preprocess_dataset(df_all, OUTPUT_DIR)
# plot_eda(df_processed, title='DentaScribe ASR Dataset')
# train, val, test = split_and_save(df_processed, OUTPUT_DIR)
# print('Data engineering complete. Clean splits ready for Andy (Role 1).')

print('[Section 6] Full pipeline ready. Download datasets and uncomment to run.')
